# Connecting to the Prompt Hub

We can connect our application to LangSmith's Prompt Hub, which will allow us to test and iterate on our prompts within LangSmith, and pull our improvements directly into our application.

### Setup

In [1]:
import os
os.environ["OPENAI_API_KEY"] = ""
os.environ["LANGSMITH_API_KEY"] = ""
os.environ["LANGSMITH_TRACING"] = ""
os.environ["LANGSMITH_PROJECT"] = "langsmith-academy"  # If you don't set this, traces will go to the Default project

In [2]:
# Or you can use a .env file
from dotenv import load_dotenv
load_dotenv(dotenv_path="../../.env", override=True)

False

### Pull a prompt from Prompt Hub

Pull in a prompt from Prompt Hub by pasting in the code snippet from the UI.

In [3]:
# Create a LangSmith API in Settings > API Keys
# Make sure API key env var is set:
# import os; os.environ["LANGSMITH_API_KEY"] = "<your-api-key>"
from langsmith import Client

client = Client()
prompt = client.pull_prompt(
    "pirate-freind"
)

Let's see what we pulled - note that we did not get the model, so this is just a StructuredPrompt and not runnable.

In [4]:
prompt

StructuredPrompt(input_variables=['language', 'question'], input_types={}, partial_variables={}, metadata={'lc_hub_owner': '-', 'lc_hub_repo': 'pirate-freind', 'lc_hub_commit_hash': '30f5cd34f9471e29dcf72596f234b9a8250130244f2c85bdf316aae5677d6ec5'}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['language'], input_types={}, partial_variables={}, template='You are a pirate from 1600s, you only speak {language}'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, template='{question}'), additional_kwargs={})], schema_={'title': 'answer', 'description': 'Extract the answer', 'type': 'object', 'properties': {'answer': {'type': 'string', 'description': 'Tge answer from LLM to the user'}}, 'required': ['answer'], 'additionalProperties': False, 'strict': True}, structured_output_kwargs={})

Cool! Now let's hydrate our prompt by calling .invoke() with our inputs

In [5]:
hydrated_prompt = prompt.invoke({"question": "Are you a captain yet?", "language": "Spanish"})
hydrated_prompt

ChatPromptValue(messages=[SystemMessage(content='You are a pirate from 1600s, you only speak Spanish', additional_kwargs={}, response_metadata={}), HumanMessage(content='Are you a captain yet?', additional_kwargs={}, response_metadata={})])

And now let's pass those messages to OpenAI and see what we get back!

In [6]:
from openai import OpenAI
from langsmith.client import convert_prompt_to_openai_format

openai_client = OpenAI()

# NOTE: We can use this utility from LangSmith to convert our hydrated prompt to openai format
converted_messages = convert_prompt_to_openai_format(hydrated_prompt)["messages"]

openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=converted_messages,
    )

ChatCompletion(id='chatcmpl-EAZzlogznAX1j0SdspOmLWVxoe7z3', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='¡Argh! No soy capitán aún, pero navego los mares con valentía y deseo alcanzar la gloria de ser uno. ¡La aventura me llama! ¿Tienes mapas del tesoro?', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1786190353, model='gpt-4o-mini-2024-07-18', object='chat.completion', moderation=None, service_tier='default', system_fingerprint='fp_ab0a2ab924', usage=CompletionUsage(completion_tokens=41, prompt_tokens=31, total_tokens=72, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cache_write_tokens=None, cached_tokens=0)))

##### [Extra: LangChain Only] Pulling down the Model Configuration

We can also pull down the saved model configuration as a LangChain RunnableBinding when we use `include_model=True`. This allows us to run our prompt template directly with the saved model configuration.

In [7]:
# Create a LangSmith API in Settings > API Keys
# Make sure API key env var is set:
# import os; os.environ["LANGSMITH_API_KEY"] = "<your-api-key>"
from langsmith import Client

client = Client()
prompt = client.pull_prompt(
    "pirate-freind",
    include_model=True
)

/usr/local/lib/python3.12/dist-packages/langchain_core/load/load.py:822: UserWarning: WARNING! extra_headers is not default parameter.
                extra_headers was transferred to model_kwargs.
                Please confirm that extra_headers is what you intended.
  loaded_obj = {k: _load(v) for k, v in obj.items()}


In [8]:
prompt

StructuredPrompt(input_variables=['language', 'question'], input_types={}, partial_variables={}, metadata={'lc_hub_owner': '-', 'lc_hub_repo': 'pirate-freind', 'lc_hub_commit_hash': '30f5cd34f9471e29dcf72596f234b9a8250130244f2c85bdf316aae5677d6ec5'}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['language'], input_types={}, partial_variables={}, template='You are a pirate from 1600s, you only speak {language}'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, template='{question}'), additional_kwargs={})], schema_={'title': 'answer', 'description': 'Extract the answer', 'type': 'object', 'properties': {'answer': {'type': 'string', 'description': 'Tge answer from LLM to the user'}}, 'required': ['answer'], 'additionalProperties': False, 'strict': True}, structured_output_kwargs={})
| _ChatModelBinding(bound=ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.5

Test out your prompt!

In [9]:
prompt.invoke({"question": "Are you a captain yet?", "language": "Spanish"})

{'answer': '¡Argh! No soy aún capitán, pero navego los mares con audacia y astucia, buscando tesoros y aventuras. ¡Pronto seré el rey de los piratas!'}

### Pull down a specific commit

Pull down a specific commit from the Prompt Hub by pasting in the code snippet from the UI.

In [10]:
# Create a LangSmith API in Settings > API Keys
# Make sure API key env var is set:
# import os; os.environ["LANGSMITH_API_KEY"] = "<your-api-key>"
from langsmith import Client

client = Client()
prompt = client.pull_prompt(
    "pirate-freind:7736eaa0"
)

Run this commit!

In [11]:
from openai import OpenAI
from langsmith.client import convert_prompt_to_openai_format

openai_client = OpenAI()

hydrated_prompt = prompt.invoke({"question": "What is the world like?", "language": "English"})
# NOTE: We can use this utility from LangSmith to convert our hydrated prompt to openai format
converted_messages = convert_prompt_to_openai_format(hydrated_prompt)["messages"]

openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=converted_messages,
    )

ChatCompletion(id='chatcmpl-EAa61Vx0VjFxJG0cAb3z6PVl425tm', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content="Arrr, matey! The world of the year 2500 be a wild and wondrous place! Technology be advanced beyond yer wildest dreams. Skyships float like clouds, powered by the harnessin' of energy from the stars. The oceans be teemin' with life, but also with ships that sail through the waves of time itself.\n\nCivilizations have expanded beyond Earth, settlin' on distant planets and moons in the far reaches of the galaxy. Communicators known as holo-maps allow folk to speak and see each other across vast distances, makin' piracy a truly interstellar affair!\n\nThe seas be still a place of adventure, but they be danced upon by sleek vessels built with both ancient craftsmanship and futuristic tech. No longer just for treasure, the bounty on a pirate’s head might include rare resources or precious knowledge.\n\nBut beware! The law be ever vi

### Uploading Prompts

You can also easily update your prompts in the hub programmatically.



In [13]:
from langchain_core.prompts import ChatPromptTemplate
from langsmith import Client

client=Client()

french_prompt = """You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer the latest question in the conversation.

Your users can only speak French, make sure you only answer your users with French.

Conversation: {conversation}
Context: {context}
Question: {question}
Answer:"""

french_prompt_template = ChatPromptTemplate.from_template(french_prompt)
client.push_prompt("french-rag-prompt", object=french_prompt_template)

'https://smith.langchain.com/prompts/french-rag-prompt/75567b82?organizationId=a97c2bb7-6336-4e15-9372-18edb5a40ca1'

You can also push a prompt as a RunnableSequence of a prompt and a model. This is useful for storing the model configuration you want to use with this prompt. The provider must be supported by the LangSmith playground.

In [14]:
from langchain_core.prompts import ChatPromptTemplate
from langsmith import Client
from langchain_openai import ChatOpenAI

client=Client()
model = ChatOpenAI(model="gpt-4o-mini")

french_prompt = """You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer the latest question in the conversation.

Your users can only speak French, make sure you only answer your users with French.

Conversation: {conversation}
Context: {context}
Question: {question}
Answer:"""
french_prompt_template = ChatPromptTemplate.from_template(french_prompt)
chain = french_prompt_template | model
client.push_prompt("french-runnable-sequence", object=chain)

LangSmithConflictError: Conflict for /commits/-/french-runnable-sequence. HTTPError('409 Client Error: Conflict for url: https://api.smith.langchain.com/commits/-/french-runnable-sequence', '{"error":"Nothing to commit: prompt has not changed since latest commit"}\n')